Import relevant packages

In [71]:
import numpy as np # this is our standard 'do some efficient matrix maths' package
import matplotlib.pyplot as plt # this is our standard plotting package
import glob # help us list files and recognise pathnames
import os # this package provides general functionality for our operating systems
import ee # this is the earthengine-api package
import geemap.foliumap as emap # this imports a method of creating interactive maps with geemap
import requests # requests is a HTTP library and so it will enable us to send out HTTP requests to get data
import rasterio # Enable us to work efficeintly with gridded raster datasets in python
from pyproj import Geod # pyproj helps us work with coordinate transformations and projections
import logging
logging.getLogger('rasterio').setLevel(logging.ERROR)

Authorise Google Earth Engine

In [72]:
ee.Authenticate()
ee.Initialize(project="uksa-training-course-materials")

Obtain data from Google Earth Engine

In [73]:
# we define the path
path='/home/jovyan/data/Tutorial_data/scratch/GEE_test/roblar/'

# check if the directory already exists
isExist = os.path.exists(path)
if not isExist:
   # Create a new directory because it does not exist
   os.makedirs(path) #we create the directory
   print("Created directory!")

Download the data collection

In [97]:
def obtain_data(region, start_date, end_date, bands, collection_of_interest, include_end_date=False, max_cloud_cover=15):
    """
    region (geometry): region of interest
    start_date (str): Starting year for the Sentinel-2 imagery.
    end_data (str): Ending year for the Sentinel-2 imagery.
    bands(list): Bands to select from the surface reflectance product B2, B3, B4, and B8 correspond to Blue, Green, Red, and NIR.
    collection_of_interest (str): Earth ENgine Catalogue Data collection of interest
    max_cloud_cover (float): Max percentage of cloud coverage allowed - if the cloud coverage is higher, the imgage is not downloaded
    """

    start_date = ee.Date(start_date)

    if include_end_date:
        end_date = ee.Date(end_date)
    else:
        end_date = ee.Date(end_date).advance(-1, "day")

    # Filter input collections by desired date range, region and cloud coverage.
    criteria  = ee.Filter.And(
        ee.Filter.geometry(region),
        ee.Filter.date(start_date, end_date)
    )
    #'CLOUD_COVER' for Landsat.
    SAT = ee.ImageCollection(collection_of_interest) \
                      .filter(criteria) \
                      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud_cover))
    #Selects up to 5 images with the lowest cloud coverage.
    SAT = SAT.sort('CLOUDY_PIXEL_PERCENTAGE').limit(5).select(bands)
    
    image_collections = {
        'SAT': SAT,
        }

    return image_collections

In [98]:
def download_data(filepath, region, scale_dict, start_date, end_date, bands, collection_of_interest, \
                  desired_collections=None, include_end_date=False, max_cloud_cover=15, overwrite=False):
    """
    filepath (str): Path to the directory where the data will be stored
    region (geometry): region of interest
    scale_dict (str): spatial resolution requested
    start_date (str): Starting year for the Sentinel-2 imagery.
    end_date (str): Ending year for the Sentinel-2 imagery.
    bands(list): Bands to select from the surface reflectance product B2, B3, B4, and B8 correspond to Blue, Green, Red, and NIR.
    collection_of_interest (str): Earth Eggine Catalogue Data collection of interest
    include_end_date (bool): either False or True
    max_cloud_cover (float): max percentage of cloud coverage allowed - if the cloud coverage is higher, the img is not downloaded
    overwrite(bool): either False or True
    """

    # Obtains all image collections defined in the request function for
    # the chosen test district and date range, with maximum n% cloud cover.
    image_collections = obtain_data(
        region,
        start_date,
        end_date,
        bands,
        collection_of_interest,
        include_end_date,
        max_cloud_cover,
    )

    # Filters out unwanted collections if defined above.
    if desired_collections:
        image_collections = {
            collection: image_collections[collection] \
            for collection in desired_collections
        }

    # Creates a subfolder in the base directory for the start date
    out_dir = f'{filepath}/{start_date}'
    if not os.path.isdir(out_dir):
        os.mkdir(out_dir)

    # Iterating through each image collection.
    for collection_name, collection in image_collections.items():
        print(collection_name)
        # Counts the number of images in a collection.
        collection_size = collection.size().getInfo()

        # Skips the image collection if it contains no images.
        if collection_size == 0:
            print('No images in collection, skipping.')
            continue

        # Creates additional subfolders for each image collection.
        collection_dir = f'{out_dir}/{collection_name}'
        if not os.path.isdir(collection_dir):
            os.mkdir(collection_dir)

        # Counts number of .tif files already in image collection subfolder.
        tif_count = len(glob.glob1(collection_dir,"*.tif"))

        # Assumes the download for this collection is already complete and
        # therefore skips, provided the number of .tif files already in
        # chosen directory matches the number of images in the collection
        # and overwrite is set to False.
        if collection_size == tif_count and overwrite == False:
            print('Correct number of .tif files for image collection already in directory, skipping.')
            continue

        # Exports each image in the filtered image collection to
        # geoTIFF format.
        emap.ee_export_image_collection(
            collection,
            collection_dir,
            crs='EPSG:4326',
            scale=scale_dict[collection_name],
            region=region
        )

In [99]:
#Creation of Google Drive Folder
filepath = '/home/jovyan/data/Tutorial_data/scratch/GEE_test/roblar/S2/'
# check if the directory already exists
isExist = os.path.exists(filepath)
if not isExist:
   # Create a new directory because it does not exist
   os.makedirs(filepath) #we create the directory
   print("Created directory!")

In [100]:
#Bounding box for Baihetan Dam
region = ee.Geometry.Rectangle([102.85, 27.1, 102.95, 27.3])

# Defines the desired pixel scale for each image, have set to the native resolution of each satellite.
scale_dict = {
    'SAT': 10 #Appropriate for B2, B3, B4, and B8
}

In [101]:
#Select desired dates
years = ["2017","2018","2019","2020","2021","2022","2023", "2024"] #Years Sentinal-2 data is fully available for
months = ["01", "04", "07", "10"] #Every 3 months
#Creates a list of start and end dates
date_list=[]
date_list = [f"{year}-{month}-01" for year in years for month in months]
date_list.append("2025-01-01") #This will be the last date in dates
date_list.append("2025-04-01") #This will not be included in dates
dates = [(date_list[i], date_list[i+1]) for i in range(len(date_list)-1)][0:-1]

dates

[('2017-01-01', '2017-04-01'),
 ('2017-04-01', '2017-07-01'),
 ('2017-07-01', '2017-10-01'),
 ('2017-10-01', '2018-01-01'),
 ('2018-01-01', '2018-04-01'),
 ('2018-04-01', '2018-07-01'),
 ('2018-07-01', '2018-10-01'),
 ('2018-10-01', '2019-01-01'),
 ('2019-01-01', '2019-04-01'),
 ('2019-04-01', '2019-07-01'),
 ('2019-07-01', '2019-10-01'),
 ('2019-10-01', '2020-01-01'),
 ('2020-01-01', '2020-04-01'),
 ('2020-04-01', '2020-07-01'),
 ('2020-07-01', '2020-10-01'),
 ('2020-10-01', '2021-01-01'),
 ('2021-01-01', '2021-04-01'),
 ('2021-04-01', '2021-07-01'),
 ('2021-07-01', '2021-10-01'),
 ('2021-10-01', '2022-01-01'),
 ('2022-01-01', '2022-04-01'),
 ('2022-04-01', '2022-07-01'),
 ('2022-07-01', '2022-10-01'),
 ('2022-10-01', '2023-01-01'),
 ('2023-01-01', '2023-04-01'),
 ('2023-04-01', '2023-07-01'),
 ('2023-07-01', '2023-10-01'),
 ('2023-10-01', '2024-01-01'),
 ('2024-01-01', '2024-04-01'),
 ('2024-04-01', '2024-07-01'),
 ('2024-07-01', '2024-10-01'),
 ('2024-10-01', '2025-01-01')]

In [102]:
for start_date, end_date in dates:
    print(start_date, end_date)
    download_data(
        filepath=filepath,
        region=region,
        scale_dict=scale_dict,
        start_date=start_date,
        end_date=end_date,
        include_end_date=False,
        max_cloud_cover=15,
        bands=['B2', 'B3', 'B4', 'B8'], #Blue, Green, Red, and NIR
        collection_of_interest="COPERNICUS/S2_SR_HARMONIZED",
        desired_collections=None,
        overwrite=True
    )

2017-01-01 2017-04-01
SAT
No images in collection, skipping.
2017-04-01 2017-07-01
SAT
No images in collection, skipping.
2017-07-01 2017-10-01
SAT
No images in collection, skipping.
2017-10-01 2018-01-01
SAT
No images in collection, skipping.
2018-01-01 2018-04-01
SAT
Total number of images: 4

Exporting 1/4: /home/jovyan/data/Tutorial_data/scratch/GEE_test/roblar/S2//2018-01-01/SAT/20180219T034759_20180219T034801_T48RTR.tif
Generating URL ...
Please wait ...
Data downloaded to /home/jovyan/data/Tutorial_data/scratch/GEE_test/roblar/S2/2018-01-01/SAT/20180219T034759_20180219T034801_T48RTR.tif


Exporting 2/4: /home/jovyan/data/Tutorial_data/scratch/GEE_test/roblar/S2//2018-01-01/SAT/20180219T034759_20180219T034801_T48RTQ.tif
Generating URL ...
Please wait ...
Data downloaded to /home/jovyan/data/Tutorial_data/scratch/GEE_test/roblar/S2/2018-01-01/SAT/20180219T034759_20180219T034801_T48RTQ.tif


Exporting 3/4: /home/jovyan/data/Tutorial_data/scratch/GEE_test/roblar/S2//2018-01-01/SAT/2